Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline
Build a 2-step pipeline. Step 1 extracts the research question, method, and key finding from a paper abstract as structured data. Step 2 generates a plain-language summary for a non-expert audience, using only the structured version from Step 1, not the original abstract.

Input: abstract_text

Output: the structured extraction from Step 1, and the layperson summary from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: research_question, method, key_finding)

Step 2 — Template: ChatPromptTemplate (system persona: "science communicator explaining to a general audience")  |  Parser: StrOutputParser

In [1]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

## Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: research_question, method, key_finding)


In [2]:
class ResearchData(BaseModel):
    research_question: str = Field(
        description="The main question the research investigates"
    )

    method: str = Field(
        description="The research method, study design, or procedure"
    )

    key_finding: str = Field(
        description="The main finding reported in the abstract"
    )

In [3]:
research_parser = PydanticOutputParser(
    pydantic_object=ResearchData
)

In [4]:
print(research_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"research_question": {"description": "The main question the research investigates", "title": "Research Question", "type": "string"}, "method": {"description": "The research method, study design, or procedure", "title": "Method", "type": "string"}, "key_finding": {"description": "The main finding reported in the abstract", "title": "Key Finding", "type": "string"}}, "required": ["research_question", "method", "key_finding"]}
```


In [5]:
research_prompt = PromptTemplate(
    template="""
You are extracting information from a research paper abstract.

Extract the following fields:
1. research_question
2. method
3. key_finding

Rules:
- Use only information supported by the abstract.
- Do not invent research methods or findings.
- If information is not provided, clearly indicate that it
  is not specified in the abstract.
- Keep each field concise and accurate.

Research abstract:
{abstract_text}

{format_instructions}
""",
    input_variables=["abstract_text"],
    partial_variables={
        "format_instructions":
            research_parser.get_format_instructions()
    }
)

In [7]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [8]:
research_step1_chain = (
    research_prompt
    | llm
    | research_parser
)

In [9]:
abstract_text = """
Researchers investigated whether regular walking improves
sleep quality in adults. They conducted an eight-week study.
One group walked for 30 minutes daily, while another group
maintained its usual routine. The walking group reported
improved sleep quality compared with the other group.
"""

research_data = research_step1_chain.invoke({
    "abstract_text": abstract_text
})

print(research_data)

research_question='Does regular walking improve sleep quality in adults?' method='An eight‑week study comparing a group that walked 30 minutes daily with a control group that maintained its usual routine.' key_finding='The walking group reported improved sleep quality compared with the control group.'


In [10]:
print("Research question:", research_data.research_question)
print("Method:", research_data.method)
print("Key finding:", research_data.key_finding)

Research question: Does regular walking improve sleep quality in adults?
Method: An eight‑week study comparing a group that walked 30 minutes daily with a control group that maintained its usual routine.
Key finding: The walking group reported improved sleep quality compared with the control group.


## Step 2 — Template: ChatPromptTemplate (system persona: "science communicator explaining to a general audience")  |  Parser: StrOutputParser

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [12]:
research_summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a science communicator explaining research
        to a general audience.

        Explain the research in simple, everyday language.

        Requirements:
        - Explain what the researchers investigated.
        - Describe what they did in understandable terms.
        - Explain the main finding clearly.
        - Avoid unnecessary scientific jargon.
        - Do not invent results or details.
        - Do not claim that a finding proves causation
          unless the provided information supports that claim.
        """
    ),
    (
        "user",
        """
        Use only the following structured research data
        to write a plain-language summary.

        Research data:
        {research_data}

        Write one clear, easy-to-understand paragraph.
        """
    )
])

In [13]:
research_summary_parser = StrOutputParser()

research_step2_chain = (
    research_summary_prompt
    | llm
    | research_summary_parser
)

In [14]:
layperson_summary = research_step2_chain.invoke({
    "research_data": research_data.model_dump_json(indent=2)
})

print(layperson_summary)

Researchers wanted to find out if walking regularly could help adults sleep better. They ran an eight‑week study in which one group walked for 30 minutes every day, while a second group kept doing what they normally did. After the study, the people who walked reported that their sleep quality had improved compared with those who didn’t walk.


In [15]:
def summarize_research_abstract(abstract_text: str):
    # Step 1: Extract structured research information
    research_data = research_step1_chain.invoke({
        "abstract_text": abstract_text
    })

    # Step 2: Explain the extracted information simply
    layperson_summary = research_step2_chain.invoke({
        "research_data": research_data.model_dump_json(
            indent=2
        )
    })

    return {
        "research_data": research_data,
        "layperson_summary": layperson_summary
    }

In [16]:
result = summarize_research_abstract(abstract_text)

print("STEP 1: STRUCTURED RESEARCH DATA")
print(result["research_data"].model_dump_json(indent=2))

print("\nSTEP 2: LAYPERSON SUMMARY")
print(result["layperson_summary"])

STEP 1: STRUCTURED RESEARCH DATA
{
  "research_question": "Does regular walking improve sleep quality in adults?",
  "method": "An eight‑week study comparing a group that walked 30 minutes daily with a control group that maintained its usual routine.",
  "key_finding": "The walking group reported improved sleep quality compared with the control group."
}

STEP 2: LAYPERSON SUMMARY
Researchers wanted to find out if walking regularly could help adults sleep better. They ran an eight‑week study in which one group walked for 30 minutes every day, while a second group kept doing what they normally did. After the study, the people who walked reported that their sleep quality had improved compared with those who didn’t walk.
